# Test: pseudo-labeling Person boxes on ppe_detection_m

`data/ppe_detection_m` has 8 real PPE-item classes (safety-helmet, safety-glove,
safety-shoes, welding-glass, and their `no-*` negatives) but **no Person class at
all** — same structural gap as the Altec PPE run, and the reason
`yolo26s_Altec_PPE_100e` can't drive a compliance verdict on its own (see
Model Comparison / Model Performance in the app).

This notebook tests whether a fresh, COCO-pretrained YOLO26 detector — which
already knows what a person looks like, just not PPE — can fill that gap: draw
Person boxes on a **sample** of ppe_detection_m's images, merge them with the
existing PPE labels, and write the result to a **new folder**. Nothing in
`data/ppe_detection_m/` is read for writing or modified — only copied from.

Once you've reviewed the sample (`preview/` below) and you're happy with the
quality, the same loop can be scaled up to the full ~15k images and used to
retrain YOLO26 on ppe_detection_m + a real Person class.

Your colleague's `YOLO26_dataset_merged.ipynb` (cells 8-10) prototyped a related
idea — pairing a person model *and* a pose model *and* the PPE model live, at
inference time, merged via `ppe_association.from_ultralytics_multi()`, to score
compliance directly. That's still available here as an optional second pass
(`USE_POSE_SECOND_PASS` below) if the single-model pass misses too many people —
but the main point of *this* notebook is different: bake the person boxes into
the **labels** once, offline, so a single retrained model learns to detect
Person itself, rather than needing two models glued together at inference time
every time the app runs.

## 1 · Config — everything you'd want to tweak lives here

In [1]:
import random
import shutil
from pathlib import Path

import yaml
import pandas as pd
from PIL import Image, ImageDraw
from ultralytics import YOLO

REPO_ROOT = Path.cwd()  # run this notebook from the repo root (P0-safety/)
SRC_DIR = REPO_ROOT / "data" / "ppe_detection_m"
SPLIT = "train"  # which split to sample from: train / valid / test
OUT_DIR = REPO_ROOT / "experiments" / "person_pseudolabel_test"

SAMPLE_N = 100  # how many images to test on first — bump this (or loop all splits) once you trust it
SEED = 42

PERSON_MODEL_PATH = REPO_ROOT / "yolo26n.pt"  # fresh COCO checkpoint — still has "person" (class 0),
                                               # unmodified by any fine-tuning
PERSON_CONF = 0.50  # deliberately higher than the app's own 0.35 slider default — these boxes become
                     # TRAINING labels, not just a live overlay a person reviews, so a false positive
                     # here is more costly than one shown on screen.

# Optional second pass with the COCO-pose checkpoint, merged the same way your colleague's notebook
# does it — catches some people the plain detection head misses. Off by default: start with the
# single-model pass, check preview/, and only turn this on if recall looks too low.
USE_POSE_SECOND_PASS = False
POSE_MODEL_PATH = REPO_ROOT / "yolo26s-pose.pt"
DEDUPE_IOU = 0.4  # passed to ppe_association.from_ultralytics_multi to avoid double-counting
                  # one real person as two boxes when both models agree on them

random.seed(SEED)
assert PERSON_MODEL_PATH.exists(), f"missing {PERSON_MODEL_PATH} — expected at repo root"
assert SRC_DIR.exists(), f"missing {SRC_DIR} — is data/ppe_detection_m checked out?"

## 2 · Read ppe_detection_m's own class list — don't hardcode it

In [2]:
with open(SRC_DIR / "dataset.yaml") as f:
    src_cfg = yaml.safe_load(f)

ORIG_CLASS_NAMES = src_cfg["names"]  # 8 classes, no "person"
PERSON_CLASS_IDX = len(ORIG_CLASS_NAMES)  # new class appended at the end -> index 8
NEW_CLASS_NAMES = ORIG_CLASS_NAMES + ["person"]

print(f"{len(ORIG_CLASS_NAMES)} existing classes: {ORIG_CLASS_NAMES}")
print(f"person will be added as class {PERSON_CLASS_IDX}")

8 existing classes: ['no-safety-glove', 'no-safety-helmet', 'no-safety-shoes', 'no-welding-glass', 'safety-glove', 'safety-helmet', 'safety-shoes', 'welding-glass']
person will be added as class 8


## 3 · Sample images, set up the output folder

Everything from here on is written under `OUT_DIR` only.

In [3]:
img_dir = SRC_DIR / SPLIT / "images"
lbl_dir = SRC_DIR / SPLIT / "labels"

all_images = sorted(img_dir.glob("*"))
sample_images = random.sample(all_images, min(SAMPLE_N, len(all_images)))
print(f"sampled {len(sample_images)} of {len(all_images)} images from {SPLIT}/images")

for sub in ("images", "labels", "preview"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

sampled 100 of 8691 images from train/images


## 4 · Helpers — reading existing labels, drawing a review image

In [4]:
import colorsys


def _class_colors(n):
    return [tuple(int(c * 255) for c in colorsys.hsv_to_rgb(i / n, 0.65, 0.95)) for i in range(n)]


CLASS_COLORS = _class_colors(len(NEW_CLASS_NAMES))
PERSON_COLOR = (255, 0, 0)  # always red, regardless of the palette above, so it stands out


def read_yolo_labels(label_path):
    """[(class_id, cx, cy, w, h), ...] straight from a YOLO-format .txt, all normalised 0-1."""
    if not label_path.exists():
        return []
    rows = []
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if not parts:
            continue
        cls_id, cx, cy, w, h = int(parts[0]), *map(float, parts[1:5])
        rows.append((cls_id, cx, cy, w, h))
    return rows


def draw_preview(image, existing_rows, person_boxes_norm):
    """existing_rows: [(class_id, cx, cy, w, h)] from the ORIGINAL ppe_detection_m label file.
    person_boxes_norm: [(cx, cy, w, h, conf)] newly detected, not yet in any label file."""
    img = image.convert("RGB").copy()
    draw = ImageDraw.Draw(img)
    w_img, h_img = img.size

    for cls_id, cx, cy, w, h in existing_rows:
        x1, y1 = (cx - w / 2) * w_img, (cy - h / 2) * h_img
        x2, y2 = (cx + w / 2) * w_img, (cy + h / 2) * h_img
        color = CLASS_COLORS[cls_id]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        draw.text((x1 + 2, y1 + 2), NEW_CLASS_NAMES[cls_id], fill=color)

    for cx, cy, w, h, conf in person_boxes_norm:
        x1, y1 = (cx - w / 2) * w_img, (cy - h / 2) * h_img
        x2, y2 = (cx + w / 2) * w_img, (cy + h / 2) * h_img
        draw.rectangle([x1, y1, x2, y2], outline=PERSON_COLOR, width=3)
        draw.text((x1 + 2, y1 + 2), f"person {conf:.2f}", fill=PERSON_COLOR)

    return img

## 5 · Run the person detector, merge labels, save everything

One pass per sampled image: detect person boxes, append them to a copy of the existing PPE labels, save the (unmodified) image + the merged label + an annotated preview — all under `OUT_DIR`.

In [5]:
person_model = YOLO(str(PERSON_MODEL_PATH))
pose_model = YOLO(str(POSE_MODEL_PATH)) if USE_POSE_SECOND_PASS else None

if USE_POSE_SECOND_PASS:
    import sys
    sys.path.insert(0, str(REPO_ROOT))
    from ppe_association import from_ultralytics_multi

summary_rows = []

for img_path in sample_images:
    image = Image.open(img_path)
    w_img, h_img = image.size

    person_result = person_model.predict(str(img_path), classes=[0], conf=PERSON_CONF, verbose=False)[0]

    if USE_POSE_SECOND_PASS:
        pose_result = pose_model.predict(str(img_path), conf=PERSON_CONF, verbose=False)[0]
        merged = from_ultralytics_multi(person_result, pose_result, dedupe_iou=DEDUPE_IOU)
        person_boxes_norm = [
            (
                (d.x1 + d.x2) / 2 / w_img, (d.y1 + d.y2) / 2 / h_img,
                (d.x2 - d.x1) / w_img, (d.y2 - d.y1) / h_img,
                d.confidence,
            )
            for d in merged if d.label.strip().lower() == "person"
        ]
    else:
        boxes = person_result.boxes
        person_boxes_norm = [(*xywhn.tolist(), float(conf)) for xywhn, conf in zip(boxes.xywhn, boxes.conf)]

    # --- merged label file: original PPE lines untouched, plus new person lines ---
    label_path = lbl_dir / (img_path.stem + ".txt")
    existing_rows = read_yolo_labels(label_path)
    new_lines = [f"{r[0]} {r[1]:.6f} {r[2]:.6f} {r[3]:.6f} {r[4]:.6f}" for r in existing_rows]
    new_lines += [f"{PERSON_CLASS_IDX} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for cx, cy, w, h, _ in person_boxes_norm]
    (OUT_DIR / "labels" / (img_path.stem + ".txt")).write_text("\n".join(new_lines) + "\n")

    # --- unmodified image copy, ready for training ---
    shutil.copy2(img_path, OUT_DIR / "images" / img_path.name)

    # --- annotated preview, for you to eyeball ---
    preview = draw_preview(image, existing_rows, person_boxes_norm)
    preview.save(OUT_DIR / "preview" / (img_path.stem + ".jpg"), quality=85)

    confs = [c for *_, c in person_boxes_norm]
    summary_rows.append({
        "image": img_path.name,
        "n_existing_ppe_boxes": len(existing_rows),
        "n_person_detected": len(person_boxes_norm),
        "min_person_conf": round(min(confs), 3) if confs else None,
        "max_person_conf": round(max(confs), 3) if confs else None,
    })

print(f"done \u2014 wrote {len(sample_images)} images to {OUT_DIR}")

done — wrote 100 images to /Users/joanna/code/ds-final-project/P0-safety/experiments/person_pseudolabel_test


## 6 · Summary — what to check before trusting any of this

In [6]:
summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT_DIR / "summary.csv", index=False)

zero_person = summary[summary["n_person_detected"] == 0]
print(summary.describe(include="all"))
print()
print(f"{len(zero_person)}/{len(summary)} sampled images got NO person box at all \u2014 check these first:")
print(zero_person["image"].tolist())

# Reference data.yaml for this sample only \u2014 NOT wired up for a real training run yet: only
# SAMPLE_N images have both PPE + person labels here, and they aren't split into train/valid/test.
# Scale up (loop over every image in every split, drop SAMPLE_N) once preview/ looks good.
new_cfg = {
    "path": str(OUT_DIR.resolve()),
    "train": "images",
    "val": "images",
    "nc": len(NEW_CLASS_NAMES),
    "names": NEW_CLASS_NAMES,
}
with open(OUT_DIR / "data.yaml", "w") as f:
    yaml.safe_dump(new_cfg, f)
print(f"\nwrote {OUT_DIR / 'data.yaml'} (reference only \u2014 see note above before training on it)")

                                                    image  \
count                                                 100   
unique                                                100   
top     frame00051_png.rf.54f3e9f9d2109fb50b8ee4b5ec74...   
freq                                                    1   
mean                                                  NaN   
std                                                   NaN   
min                                                   NaN   
25%                                                   NaN   
50%                                                   NaN   
75%                                                   NaN   
max                                                   NaN   

        n_existing_ppe_boxes  n_person_detected  min_person_conf  \
count             100.000000         100.000000        90.000000   
unique                   NaN                NaN              NaN   
top                      NaN                NaN              Na

## 7 · Reviewing and scaling up

- **Look at `experiments/person_pseudolabel_test/preview/*.jpg` first.** Each shows the
  original PPE boxes (per-class colors, matching `NEW_CLASS_NAMES`) plus the new person
  boxes in red with their confidence. `summary.csv` flags the zero-detection images so you
  don't have to scan all of them.
- **These are unverified auto-detections, not ground truth.** A false person box, or a
  missed one, becomes a wrong training label if you skip straight to retraining. Spot-check
  a good chunk of `preview/` — especially the zero-detection and lowest-confidence cases —
  before trusting this for a real run.
- **`PERSON_CONF`** is the main lever if you're seeing too many misses (lower it) or too many
  spurious boxes (raise it).
- **`USE_POSE_SECOND_PASS = True`** adds the pose-model pass your colleague's notebook uses,
  for images the plain detector under-recalls on — try it on the same sample and compare
  `summary.csv` before/after.
- **Scaling up**: once you're happy, change `SPLIT`/`SAMPLE_N` (or loop over all three splits)
  to run this over the full ~15k images, then point a real `yolo detect train` run at the
  resulting `data.yaml` — same pattern as the training command in
  `YOLO26_dataset_merged.ipynb` (section 11), just with `NEW_CLASS_NAMES` in place of the
  9-class list used there.

## 8 · Build two comparable training samples

You now have real, reviewed pseudo-labels for `ppe_detection_m`. The next question is
whether adding them actually helps — so this section builds two small, **matched**
training sets and trains a short run on each:

- **Run 1 — "as-is":** a sample from all three datasets, unchanged. `ppe_detection_m`'s
  slice contributes no Person boxes at all, same as it is on disk today.
- **Run 2 — "pseudo-labeled":** the *same* `css-data` and `PPE Kit Detection` images as
  Run 1, and the *same* `ppe_detection_m` images too — but swapped for the pseudo-labeled
  copies from Section 5 (original PPE boxes + whatever Person box the detector found,
  including none). Images where the detector found nobody are kept, not dropped — filtering
  those out would bias this slice toward the detector's easiest cases, and `css-data` /
  `PPE Kit Detection` aren't filtered that way either, so it wouldn't be a fair comparison.

Keeping `css-data` and `PPE Kit Detection` identical between the two runs means the only
thing that differs is whether `ppe_detection_m`'s images carry a Person box — so any gap
in the results is attributable to that, not to the two runs happening to see different,
unequally-hard images.

**Unifying the class lists.** `css-data`, `PPE Kit Detection` and `ppe_detection_m` don't
share one class taxonomy (your own `YOLO8.ipynb` comparison already found this — see its
"What this says about picking a dataset" note). Rather than invent a new scheme, this
maps all three onto the five slots the app itself already tracks in
`streamlit_app/detector.py` (`SLOT_ITEMS`: hardhat, vest, mask, gloves, boots) plus
Person. Classes the app doesn't track (`Safety Cone`, `machinery`, `vehicle`, `goggles`,
`welding-glass`, `PPE Kit Detection`'s ambiguous `none`) are dropped rather than
force-mapped to something they aren't.

One side effect worth knowing about going in: `PPE Kit Detection` has a `vest` class but
no `no_vest` negative, so it can only ever contribute *positive* Vest supervision, never
a "no vest" example — same asymmetry that exists in the source data, not something
introduced here.


In [13]:
# --- sample sizes & training config -----------------------------------------------
N_CSS = 100          # sampled from css-data/train (2605 available)
N_KIT = 100          # sampled from "PPE Kit Detection"/images/train (1132 available)
VAL_FRACTION = 0.2   # applied to every source, including the ppe_detection_m slice
SPLIT_SEED = 7        # controls the train/val split, shared by both runs
SAMPLE_SEED_CSS = 101  # separate seeds per dataset so changing one sample size
SAMPLE_SEED_KIT = 102  # doesn't change what gets picked from the others

TRAIN_EPOCHS = 40     # short on purpose — this is a comparability test, not a final model
IMGSZ = 640
BATCH = 16            # lower than the 24 used for the full 100-epoch runs; friendlier to a laptop
BASE_CHECKPOINT = REPO_ROOT / "yolo26n.pt"  # same fresh COCO checkpoint both runs start from

CSS_DIR = REPO_ROOT / "data" / "css-data"
KIT_DIR = REPO_ROOT / "data" / "PPE Kit Detection"

# Read css-data's own class order rather than assuming it — same habit as ORIG_CLASS_NAMES
# above. Do NOT rely on a dict's key order to stand in for this: dict literals below are
# written in a readable grouping, not necessarily the dataset's actual id order.
with open(CSS_DIR / "data.yaml") as f:
    CSS_RAW_NAMES = yaml.safe_load(f)["names"]

MERGE_ROOT = REPO_ROOT / "experiments" / "merge_sim"
RUN1_DIR = MERGE_ROOT / "run1_as_is"
RUN2_DIR = MERGE_ROOT / "run2_pseudo_labeled"

# --- the unified class list, and how each dataset's raw names map onto it ---------
# A raw class mapped to None is dropped (its label lines are simply not written out).
UNIFIED_CLASSES = [
    "Person",
    "Hardhat", "NO-Hardhat",
    "Vest", "NO-Vest",
    "Mask", "NO-Mask",
    "Gloves", "NO-Gloves",
    "Boots", "NO-Boots",
]

CSS_DATA_MAP = {
    "Hardhat": "Hardhat", "NO-Hardhat": "NO-Hardhat",
    "Mask": "Mask", "NO-Mask": "NO-Mask",
    "Safety Vest": "Vest", "NO-Safety Vest": "NO-Vest",
    "Person": "Person",
    "Safety Cone": None, "machinery": None, "vehicle": None,
}

# Confirmed against Ultralytics' docs page for this dataset in YOLO8.ipynb (cell 14) —
# no data.yaml ships with the Kaggle download, so this order isn't a guess.
PPE_KIT_RAW_NAMES = [
    "helmet", "gloves", "vest", "boots", "goggles", "none",
    "Person", "no_helmet", "no_goggle", "no_gloves", "no_boots",
]
PPE_KIT_MAP = {
    "helmet": "Hardhat", "no_helmet": "NO-Hardhat",
    "vest": "Vest",  # no vest-negative class in this dataset, see note above
    "gloves": "Gloves", "no_gloves": "NO-Gloves",
    "boots": "Boots", "no_boots": "NO-Boots",
    "Person": "Person",
    "goggles": None, "no_goggle": None, "none": None,
}

PPE_M_MAP = {
    "safety-helmet": "Hardhat", "no-safety-helmet": "NO-Hardhat",
    "safety-glove": "Gloves", "no-safety-glove": "NO-Gloves",
    "safety-shoes": "Boots", "no-safety-shoes": "NO-Boots",
    "welding-glass": None, "no-welding-glass": None,
    "person": "Person",  # only present in the pseudo-labeled copy (Section 5), never in the original
}

print(f"{len(UNIFIED_CLASSES)} unified classes: {UNIFIED_CLASSES}")


11 unified classes: ['Person', 'Hardhat', 'NO-Hardhat', 'Vest', 'NO-Vest', 'Mask', 'NO-Mask', 'Gloves', 'NO-Gloves', 'Boots', 'NO-Boots']


In [14]:
# --- helper: remap one label file onto UNIFIED_CLASSES and copy the image ---------
UNIFIED_INDEX = {name: i for i, name in enumerate(UNIFIED_CLASSES)}


def remap_and_copy(img_path, label_path, raw_names, name_map, dst_images, dst_labels, prefix):
    """Reads a YOLO label file in its dataset's own class order, drops any class that
    maps to None, remaps the rest onto UNIFIED_CLASSES, and copies both the (untouched)
    image and the new label file into dst_images / dst_labels. `prefix` keeps filenames
    from colliding across datasets that happen to share a name like "image1.jpg"."""
    new_lines = []
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue
            cls_id, cx, cy, w, h = int(parts[0]), *parts[1:5]
            raw_name = raw_names[cls_id]
            unified_name = name_map.get(raw_name, None)
            if unified_name is None:
                continue
            new_lines.append(f"{UNIFIED_INDEX[unified_name]} {cx} {cy} {w} {h}")

    stem = f"{prefix}__{img_path.stem}"
    shutil.copy2(img_path, dst_images / f"{stem}{img_path.suffix}")
    (dst_labels / f"{stem}.txt").write_text("\n".join(new_lines) + ("\n" if new_lines else ""))


def make_run_dirs(run_dir):
    for split in ("train", "val"):
        for sub in ("images", "labels"):
            (run_dir / split / sub).mkdir(parents=True, exist_ok=True)


def write_run_yaml(run_dir):
    cfg = {
        "path": str(run_dir.resolve()),
        "train": "train/images",
        "val": "val/images",
        "nc": len(UNIFIED_CLASSES),
        "names": UNIFIED_CLASSES,
    }
    with open(run_dir / "data.yaml", "w") as f:
        yaml.safe_dump(cfg, f)
    return run_dir / "data.yaml"


In [15]:
# --- pick the samples, once, shared by both runs -----------------------------------
css_pool = sorted((CSS_DIR / "train" / "images").glob("*"))
css_sample = random.Random(SAMPLE_SEED_CSS).sample(css_pool, min(N_CSS, len(css_pool)))

kit_pool = sorted((KIT_DIR / "images" / "train").glob("*"))
kit_sample = random.Random(SAMPLE_SEED_KIT).sample(kit_pool, min(N_KIT, len(kit_pool)))

# ppe_detection_m's slice isn't re-sampled — it's exactly the pseudo-labeled set already
# built and reviewed in Section 5, including any images the detector found nobody in.
# A zero-detection image just carries its original PPE-only labels into Run 2 unchanged,
# same as it does into Run 1 — dropping it would have skewed this slice toward the
# detector's easiest cases, which the other two datasets aren't filtered for either.
ppe_m_sample = [SRC_DIR / SPLIT / "images" / name for name in summary["image"].tolist()]

print(f"css-data: {len(css_sample)} sampled")
print(f"PPE Kit Detection: {len(kit_sample)} sampled")
print(f"ppe_detection_m: {len(ppe_m_sample)} sampled (all of Section 5's output, zero-detection images included)")


def split_train_val(items, seed=SPLIT_SEED, val_fraction=VAL_FRACTION):
    shuffled = items[:]
    random.Random(seed).shuffle(shuffled)
    n_val = max(1, round(len(shuffled) * val_fraction))
    return shuffled[n_val:], shuffled[:n_val]


# Split once — reused as-is for both runs, so Run 1 and Run 2 validate on the same images too.
css_train, css_val = split_train_val(css_sample)
kit_train, kit_val = split_train_val(kit_sample)
ppe_m_train, ppe_m_val = split_train_val(ppe_m_sample)

print(f"train/val split: css {len(css_train)}/{len(css_val)}, "
      f"kit {len(kit_train)}/{len(kit_val)}, ppe_m {len(ppe_m_train)}/{len(ppe_m_val)}")


css-data: 100 sampled
PPE Kit Detection: 100 sampled
ppe_detection_m: 100 sampled (all of Section 5's output, zero-detection images included)
train/val split: css 80/20, kit 80/20, ppe_m 80/20


In [16]:
# --- Run 1: "as-is" — ppe_detection_m contributes no Person boxes ------------------
make_run_dirs(RUN1_DIR)

for split_name, css_imgs, kit_imgs, ppe_m_imgs in (
    ("train", css_train, kit_train, ppe_m_train),
    ("val", css_val, kit_val, ppe_m_val),
):
    dst_images, dst_labels = RUN1_DIR / split_name / "images", RUN1_DIR / split_name / "labels"

    for img in css_imgs:
        lbl = CSS_DIR / "train" / "labels" / (img.stem + ".txt")
        remap_and_copy(img, lbl, CSS_RAW_NAMES, CSS_DATA_MAP,
                        dst_images, dst_labels, prefix="cssdata")

    for img in kit_imgs:
        lbl = KIT_DIR / "labels" / "train" / (img.stem + ".txt")
        remap_and_copy(img, lbl, PPE_KIT_RAW_NAMES, PPE_KIT_MAP,
                        dst_images, dst_labels, prefix="ppekit")

    for img in ppe_m_imgs:
        lbl = SRC_DIR / SPLIT / "labels" / (img.stem + ".txt")  # ORIGINAL labels — no person
        remap_and_copy(img, lbl, ORIG_CLASS_NAMES, PPE_M_MAP,
                        dst_images, dst_labels, prefix="ppem")

run1_yaml = write_run_yaml(RUN1_DIR)
print(f"Run 1 written to {RUN1_DIR}")
print(f"  train: {len(list((RUN1_DIR/'train'/'images').glob('*')))} images")
print(f"  val:   {len(list((RUN1_DIR/'val'/'images').glob('*')))} images")


Run 1 written to /Users/joanna/code/ds-final-project/P0-safety/experiments/merge_sim/run1_as_is
  train: 242 images
  val:   64 images


In [17]:
# --- Run 2: "pseudo-labeled" — same css-data/PPE Kit Detection images as Run 1, -----
# --- but ppe_detection_m's slice now carries the added Person box ------------------
make_run_dirs(RUN2_DIR)

PSEUDO_IMAGES_DIR = OUT_DIR / "images"
PSEUDO_LABELS_DIR = OUT_DIR / "labels"

for split_name, css_imgs, kit_imgs, ppe_m_imgs in (
    ("train", css_train, kit_train, ppe_m_train),
    ("val", css_val, kit_val, ppe_m_val),
):
    dst_images, dst_labels = RUN2_DIR / split_name / "images", RUN2_DIR / split_name / "labels"

    for img in css_imgs:  # identical to Run 1
        lbl = CSS_DIR / "train" / "labels" / (img.stem + ".txt")
        remap_and_copy(img, lbl, CSS_RAW_NAMES, CSS_DATA_MAP,
                        dst_images, dst_labels, prefix="cssdata")

    for img in kit_imgs:  # identical to Run 1
        lbl = KIT_DIR / "labels" / "train" / (img.stem + ".txt")
        remap_and_copy(img, lbl, PPE_KIT_RAW_NAMES, PPE_KIT_MAP,
                        dst_images, dst_labels, prefix="ppekit")

    for img in ppe_m_imgs:  # <-- the only thing that differs from Run 1
        pseudo_img = PSEUDO_IMAGES_DIR / img.name       # same pixels as Run 1's copy
        pseudo_lbl = PSEUDO_LABELS_DIR / (img.stem + ".txt")  # PPE boxes + Person box
        remap_and_copy(pseudo_img, pseudo_lbl, NEW_CLASS_NAMES, PPE_M_MAP,
                        dst_images, dst_labels, prefix="ppem")

run2_yaml = write_run_yaml(RUN2_DIR)
print(f"Run 2 written to {RUN2_DIR}")
print(f"  train: {len(list((RUN2_DIR/'train'/'images').glob('*')))} images")
print(f"  val:   {len(list((RUN2_DIR/'val'/'images').glob('*')))} images")


Run 2 written to /Users/joanna/code/ds-final-project/P0-safety/experiments/merge_sim/run2_pseudo_labeled
  train: 242 images
  val:   64 images


## 9 · Train both models

Same fresh `yolo26s.pt` checkpoint, same epochs/image size/batch, same train/val split
sizes — the only difference between the two calls is which `data.yaml` they point at.
`device` is left unset so Ultralytics picks the best backend available on this machine
(MPS on Apple Silicon, otherwise CPU) — the `device: '0'` you'll see in the existing
100-epoch runs' `args.yaml` is your colleague's CUDA GPU and won't apply here.

At ~150-180 train images each and 40 epochs, expect this to take a while on a laptop —
likely longer than the pseudo-labeling pass in Section 5, since that only ran inference
while this actually trains. Let both `model.train()` calls finish before reading
results in Section 10.


In [ ]:
run1_model = YOLO(str(BASE_CHECKPOINT))
run1_train_results = run1_model.train(
    data=str(run1_yaml),
    epochs=TRAIN_EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(REPO_ROOT / "runs" / "detect"),
    name="merge_sim_run1_as_is",
    exist_ok=True,
)


New https://pypi.org/project/ultralytics/8.4.130 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.127 🚀 Python-3.12.9 torch-2.13.0 CPU (Apple M4 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/joanna/code/ds-final-project/P0-safety/experiments/merge_sim/run1_as_is/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_r

In [ ]:
run2_model = YOLO(str(BASE_CHECKPOINT))
run2_train_results = run2_model.train(
    data=str(run2_yaml),
    epochs=TRAIN_EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(REPO_ROOT / "runs" / "detect"),
    name="merge_sim_run2_pseudo_labeled",
    exist_ok=True,
)


## 10 · Compare results

Both runs share `UNIFIED_CLASSES`, so — unlike the per-class table on the app's Model
Performance page, which has to leave cells blank where models don't share a class —
these two are directly comparable class-for-class, including Person.


In [ ]:
RUN1_RESULTS_DIR = REPO_ROOT / "runs" / "detect" / "merge_sim_run1_as_is"
RUN2_RESULTS_DIR = REPO_ROOT / "runs" / "detect" / "merge_sim_run2_pseudo_labeled"

def final_row(results_dir):
    df = pd.read_csv(results_dir / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]
    return {
        "epochs_run": int(last["epoch"]),
        "precision": round(last["metrics/precision(B)"], 3),
        "recall": round(last["metrics/recall(B)"], 3),
        "mAP50": round(last["metrics/mAP50(B)"], 3),
        "mAP50-95": round(last["metrics/mAP50-95(B)"], 3),
    }

comparison = pd.DataFrame({
    "Run 1 — as-is": final_row(RUN1_RESULTS_DIR),
    "Run 2 — pseudo-labeled": final_row(RUN2_RESULTS_DIR),
}).T
comparison.index.name = "run"
print(comparison)


In [ ]:
# --- confusion matrices side by side, same UNIFIED_CLASSES axes on both ------------
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
for ax, results_dir, title in (
    (axes[0], RUN1_RESULTS_DIR, "Run 1 — as-is"),
    (axes[1], RUN2_RESULTS_DIR, "Run 2 — pseudo-labeled"),
):
    cm_path = results_dir / "confusion_matrix_normalized.png"
    if not cm_path.exists():
        cm_path = results_dir / "confusion_matrix.png"
    ax.imshow(mpimg.imread(cm_path))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 11 · Reading this

- **This is a comparability test, not a verdict.** A few hundred training images and 40
  epochs (check the printed counts from Section 8) is enough to see a directional signal,
  not enough to claim a final result — treat a small gap either way as "worth investigating
  further", not proof.
- **Where to expect a difference, if there is one:** Person precision/recall, and the
  Person row/column of the confusion matrix specifically — that's the only supervision
  that changed between the two runs. The other four slots (Hardhat, Vest, Mask,
  Gloves/Boots note: Mask only has real supervision from `css-data` here, since neither
  `PPE Kit Detection` nor `ppe_detection_m` has a Mask class) get the same training
  signal in both runs, so they're mainly a sanity check that nothing else regressed.
- **Next step if Run 2 looks better:** scale up the same pipeline — rerun Section 5 over
  the full `ppe_detection_m` (drop `SAMPLE_N`, loop all three splits), rebuild Run 2's
  sample with the larger pool, and consider a longer training run before treating any
  resulting weights as a real candidate.


## 12 · Scale up — pseudo-label the entire dataset

The sample results held up, so this repeats exactly the same pass from Section 5 — same
`PERSON_MODEL_PATH`, `PERSON_CONF`, and `USE_POSE_SECOND_PASS` setting from Section 1 —
over **every image in `ppe_detection_m`**, not just a sample, and across **all three
splits** (`train`/`valid`/`test`), keeping each image in whichever split it's already in.
You had the right idea: this writes a full copy of the dataset — images unchanged, labels
updated — to a new folder. Nothing under `data/ppe_detection_m/` is touched, same rule as
the sample run.

Two things worth knowing before you run this:

- **It's ~150× the work of the sample** (15,168 images vs. 100). Inference is much lighter
  than training, but this will still take a while on a laptop CPU — the loop below prints
  progress every 500 images so you can tell it's actually moving. If it's too slow, the
  same code works in Colab too (same pattern as before: upload `data/ppe_detection_m` and
  your checkpoint to Drive, adjust the paths, run there instead — ask if you want me to
  put together a Colab version of this section specifically).
- **It needs about 2.3 GB of free disk** — same size as `ppe_detection_m` itself, since
  the images get copied, not linked (keeping this folder self-contained means you can
  upload just it later, the same way `merge_sim` got uploaded for training).


In [32]:
FULL_OUT_DIR = REPO_ROOT / "experiments" / "person_pseudolabel_full"
SPLITS_FULL = ("train", "valid", "test")  # every split, keeping ppe_detection_m's own assignment

SAVE_PREVIEWS_FULL = False  # you already reviewed preview quality at sample scale in Sections 5-7;
                             # flip this on to also spot-check a subset of the full run (adds time + disk)
PREVIEW_EVERY_FULL = 50     # only used if SAVE_PREVIEWS_FULL is True — saves 1 preview every N images

for split in SPLITS_FULL:
    for kind in ("images", "labels"):
        (FULL_OUT_DIR / split / kind).mkdir(parents=True, exist_ok=True)
    if SAVE_PREVIEWS_FULL:
        (FULL_OUT_DIR / split / "preview").mkdir(parents=True, exist_ok=True)

n_total = sum(len(list((SRC_DIR / s / "images").glob("*"))) for s in SPLITS_FULL)
print(f"about to process {n_total} images across {SPLITS_FULL} -> {FULL_OUT_DIR}")


about to process 15168 images across ('train', 'valid', 'test') -> /Users/joanna/code/ds-final-project/P0-safety/experiments/person_pseudolabel_full


In [ ]:
# # Reload fresh rather than relying on Section 5's `person_model` still being in memory —
# # makes this section safe to run on its own even after a kernel restart.
# person_model = YOLO(str(PERSON_MODEL_PATH))
# pose_model = YOLO(str(POSE_MODEL_PATH)) if USE_POSE_SECOND_PASS else None

# if USE_POSE_SECOND_PASS:
#     import sys
#     sys.path.insert(0, str(REPO_ROOT))
#     from ppe_association import from_ultralytics_multi

# full_summary_rows = []

# for split in SPLITS_FULL:
#     img_dir_full = SRC_DIR / split / "images"
#     lbl_dir_full = SRC_DIR / split / "labels"
#     split_images = sorted(img_dir_full.glob("*"))
#     print(f"--- {split}: {len(split_images)} images ---")

#     for i, img_path in enumerate(split_images):
#         image = Image.open(img_path)
#         w_img, h_img = image.size

#         person_result = person_model.predict(str(img_path), classes=[0], conf=PERSON_CONF, verbose=False)[0]

#         if USE_POSE_SECOND_PASS:
#             pose_result = pose_model.predict(str(img_path), conf=PERSON_CONF, verbose=False)[0]
#             merged = from_ultralytics_multi(person_result, pose_result, dedupe_iou=DEDUPE_IOU)
#             person_boxes_norm = [
#                 (
#                     (d.x1 + d.x2) / 2 / w_img, (d.y1 + d.y2) / 2 / h_img,
#                     (d.x2 - d.x1) / w_img, (d.y2 - d.y1) / h_img,
#                     d.confidence,
#                 )
#                 for d in merged if d.label.strip().lower() == "person"
#             ]
#         else:
#             boxes = person_result.boxes
#             person_boxes_norm = [(*xywhn.tolist(), float(conf)) for xywhn, conf in zip(boxes.xywhn, boxes.conf)]

#         # --- merged label file: original PPE lines untouched, plus new person lines ---
#         label_path = lbl_dir_full / (img_path.stem + ".txt")
#         existing_rows = read_yolo_labels(label_path)
#         new_lines = [f"{r[0]} {r[1]:.6f} {r[2]:.6f} {r[3]:.6f} {r[4]:.6f}" for r in existing_rows]
#         new_lines += [f"{PERSON_CLASS_IDX} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for cx, cy, w, h, _ in person_boxes_norm]
#         (FULL_OUT_DIR / split / "labels" / (img_path.stem + ".txt")).write_text("\n".join(new_lines) + "\n")

#         # --- unmodified image copy, ready for training ---
#         shutil.copy2(img_path, FULL_OUT_DIR / split / "images" / img_path.name)

#         # --- optional annotated preview, for a subset only ---
#         if SAVE_PREVIEWS_FULL and i % PREVIEW_EVERY_FULL == 0:
#             preview = draw_preview(image, existing_rows, person_boxes_norm)
#             preview.save(FULL_OUT_DIR / split / "preview" / (img_path.stem + ".jpg"), quality=85)

#         confs = [c for *_, c in person_boxes_norm]
#         full_summary_rows.append({
#             "split": split,
#             "image": img_path.name,
#             "n_existing_ppe_boxes": len(existing_rows),
#             "n_person_detected": len(person_boxes_norm),
#             "min_person_conf": round(min(confs), 3) if confs else None,
#             "max_person_conf": round(max(confs), 3) if confs else None,
#         })

#         if (i + 1) % 500 == 0:
#             print(f"  {split}: {i + 1}/{len(split_images)} done")

#     print(f"--- {split} done ---")

# print(f"\nfinished — wrote {len(full_summary_rows)} images to {FULL_OUT_DIR}")


--- train: 8691 images ---


In [ ]:
full_summary = pd.DataFrame(full_summary_rows)
full_summary.to_csv(FULL_OUT_DIR / "summary.csv", index=False)

print("images per split:")
print(full_summary.groupby("split").size().rename("n_images"))
print()
print("zero-person-detected count per split:")
print(full_summary.groupby("split")["n_person_detected"].apply(lambda s: (s == 0).sum()).rename("n_zero_detection"))

zero_person_full = full_summary[full_summary["n_person_detected"] == 0]
print(f"\n{len(zero_person_full)}/{len(full_summary)} images across all splits got NO person box at all.")
if not SAVE_PREVIEWS_FULL:
    print("SAVE_PREVIEWS_FULL was off, so there's no preview/ for these — worth rerunning with it on "
          "if this count looks high relative to the sample's rate.")

# This data.yaml is the real thing this time — not reference-only like Section 6's, since every
# image in every split is actually here with a matching label file.
full_cfg = {
    "path": str(FULL_OUT_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(NEW_CLASS_NAMES),
    "names": NEW_CLASS_NAMES,
}
with open(FULL_OUT_DIR / "data.yaml", "w") as f:
    yaml.safe_dump(full_cfg, f)
print(f"\nwrote {FULL_OUT_DIR / 'data.yaml'} — ready for a real training run")


## 13 · Training on the full set

Once you've spot-checked the results (`summary.csv`'s zero-detection rate, and `preview/`
if you turned `SAVE_PREVIEWS_FULL` on), the next step is the same pattern as Section 9 —
`YOLO(checkpoint).train(data=str(FULL_OUT_DIR / "data.yaml"), ...)` — just at real-run
settings this time rather than the 40-epoch comparability test (worth matching the
100-epoch/batch-24 pattern the other runs in `runs/detect/` used, hardware allowing).

Given the size (15,168 images, ~2.3 GB), this is very likely a Colab job rather than a
laptop one — same upload-to-Drive approach as `merge_sim`, just with this folder instead.
Let me know once you've reviewed the results above and I'll put together the Colab
notebook for the real training run, the same way I did for the comparability test.


## 14 · Sanity check — spot-check relabelled images

A quick visual check on `FULL_OUT_DIR`'s actual output — reads straight from the merged
label files Section 12 wrote (not the optional `preview/` folder, which was off by
default for the full run), so this shows exactly what a training run would actually see.
Every box gets drawn — original PPE boxes plus the added Person box, color-coded by
class, Person always in red.

Re-run this cell to see a different random 20 each time.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

SANITY_N = 20

all_relabelled = []
for split in SPLITS_FULL:
    split_images = sorted((FULL_OUT_DIR / split / "images").glob("*"))
    all_relabelled.extend((split, p) for p in split_images)

assert all_relabelled, f"no images found under {FULL_OUT_DIR} — did Section 12 finish running?"

sanity_sample = random.sample(all_relabelled, min(SANITY_N, len(all_relabelled)))


def draw_relabelled(image, rows):
    """rows: [(class_id, cx, cy, w, h)] read straight from a FULL_OUT_DIR label file —
    draws every box, color-coded by class, with the added Person box always in red."""
    img = image.convert("RGB").copy()
    draw = ImageDraw.Draw(img)
    w_img, h_img = img.size
    for cls_id, cx, cy, w, h in rows:
        x1, y1 = (cx - w / 2) * w_img, (cy - h / 2) * h_img
        x2, y2 = (cx + w / 2) * w_img, (cy + h / 2) * h_img
        color = PERSON_COLOR if cls_id == PERSON_CLASS_IDX else CLASS_COLORS[cls_id]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1 + 2, y1 + 2), NEW_CLASS_NAMES[cls_id], fill=color)
    return img


fig, axes = plt.subplots(4, 5, figsize=(22, 18))
for ax, (split, img_path) in zip(axes.flat, sanity_sample):
    label_path = FULL_OUT_DIR / split / "labels" / (img_path.stem + ".txt")
    rows = read_yolo_labels(label_path)
    image = Image.open(img_path)
    ax.imshow(draw_relabelled(image, rows))
    ax.set_title(f"{split}/{img_path.name}", fontsize=8)
    ax.axis("off")

for ax in axes.flat[len(sanity_sample):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 15 · Write Person boxes into `data/merged`

`data/merged` is the actual dataset `yolo26s_merged_100e` was trained on — built by
`scripts/build_dataset.py`, which pools every source's images together and re-splits with
stratified sampling, rather than keeping each source's own train/valid/test. That means a
`ppe_detection_m` image's split inside `merged` often differs from its split in
`data/ppe_detection_m` itself. `data/merged/merge_manifest.csv` is the authoritative map
back to each file's origin — this section uses it rather than guessing from filenames.

A few things specific to this dataset, confirmed from `data/merged/SOURCES.md` and a live
spot-check (not assumed):

- In the merge pipeline, `ppe_detection_m` is called **`anuragraj03`**.
- **Only 9,190 of `ppe_detection_m`'s 15,168 images (~61%) made it into `data/merged`** —
  the default build drops any image with none of `merged`'s 9 core classes present, and
  that's image *and* label both. This section only updates the ones that are actually there.
- `data/merged`'s labels use their **own 9-class scheme** (`person`, `helmet`, `gloves`,
  `boots`, `vest`, `no-helmet`, `no-gloves`, `no-boots`, `no-vest`) — different class IDs
  than `person_pseudolabel_full`'s. `welding-glass`/`no-welding-glass` aren't part of it,
  so those rows get dropped rather than remapped.
- The manifest's own split-name for `ppe_detection_m`'s original split is `val`, not
  `valid` — a one-word mismatch with the folder name Section 12 uses, handled below.

**This section overwrites files in `data/merged/labels/` in place** — the one place in
this notebook that does, since you've confirmed there's a reference copy of `merged`
elsewhere and don't need the current label files preserved. `DRY_RUN` defaults to `True`
so you can check the counts before anything actually gets written.


In [ ]:
import csv

MERGED_DIR = REPO_ROOT / "data" / "merged"
assert MERGED_DIR.exists(), f"missing {MERGED_DIR}"

# data/merged uses <split>/images + <split>/labels (matches css-data and ppe_detection_m)
def merged_split_dir(split, kind):
    return MERGED_DIR / split / kind

with open(MERGED_DIR / "data.yaml") as f:
    merged_cfg = yaml.safe_load(f)
MERGED9_NAMES = merged_cfg["names"]  # data/merged's own 9-class scheme — different IDs than ours
print(f"data/merged's own {len(MERGED9_NAMES)} classes: {MERGED9_NAMES}")

# ppe_detection_m's raw class name -> data/merged's class name; None = dropped (not one of
# merged's 9 core classes). Taken from SOURCES.md's documented mapping, confirmed against a
# live merged label file rather than assumed.
ANURAGRAJ_RAW_TO_MERGED_NAME = {
    "no-safety-glove": "no-gloves",
    "no-safety-helmet": "no-helmet",
    "no-safety-shoes": "no-boots",
    "no-welding-glass": None,
    "safety-glove": "gloves",
    "safety-helmet": "helmet",
    "safety-shoes": "boots",
    "welding-glass": None,
    "person": "person",  # the class added in Section 12
}
ANURAGRAJ_TO_MERGED_ID = {
    NEW_CLASS_NAMES.index(raw): (MERGED9_NAMES.index(name) if name is not None else None)
    for raw, name in ANURAGRAJ_RAW_TO_MERGED_NAME.items()
}
print("class id remap (our id -> merged id, None = dropped):", ANURAGRAJ_TO_MERGED_ID)

# the manifest's own split name for ppe_detection_m's ORIGINAL split doesn't match the
# folder name Section 12 uses ("val" vs "valid") — this bridges the two vocabularies
ORIGINAL_SPLIT_TO_OURS = {"train": "train", "val": "valid", "test": "test"}

with open(MERGED_DIR / "merge_manifest.csv") as f:
    manifest_rows = [r for r in csv.DictReader(f) if r["source"] == "anuragraj03"]
print(f"\n{len(manifest_rows)} of ppe_detection_m's images made it into data/merged "
      f"(the rest had none of merged's {len(MERGED9_NAMES)} core classes and were dropped entirely)")

In [ ]:
DRY_RUN = True  # flip to False once the counts below look right — this OVERWRITES files in
                 # data/merged/<split>/labels/ in place (per your call — a reference copy
                 # exists elsewhere), unlike every other section in this notebook.

n_updated = 0
n_missing_source = 0
n_rows_written = 0
n_rows_dropped = 0

for row in manifest_rows:
    our_split = ORIGINAL_SPLIT_TO_OURS[row["original_split"]]
    our_label_path = FULL_OUT_DIR / our_split / "labels" / (Path(row["original_filename"]).stem + ".txt")
    if not our_label_path.exists():
        n_missing_source += 1
        continue

    our_rows = read_yolo_labels(our_label_path)
    new_lines = []
    for cls_id, cx, cy, w, h in our_rows:
        merged_id = ANURAGRAJ_TO_MERGED_ID[cls_id]
        if merged_id is None:
            n_rows_dropped += 1
            continue
        new_lines.append(f"{merged_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        n_rows_written += 1

    target_label_path = merged_split_dir(row["split"], "labels") / (Path(row["merged_filename"]).stem + ".txt")
    if not DRY_RUN:
        target_label_path.write_text("\n".join(new_lines) + "\n")
    n_updated += 1

print(f"{'[DRY RUN] would update' if DRY_RUN else 'updated'} {n_updated} label files under {MERGED_DIR}/<split>/labels")
print(f"{n_missing_source} manifest rows had no matching file in {FULL_OUT_DIR} "
      f"(a Section 12 gap — should be 0; worth investigating if not)")
print(f"{n_rows_written} boxes written, {n_rows_dropped} dropped "
      f"(welding-glass / no-welding-glass — not part of merged's 9-class scheme)")

## 16 · Verify the write

Same idea as Section 14, pointed at `data/merged` this time — samples a few of the files
just updated and draws every box using `merged`'s own class names/colors, reading straight
from what's now on disk. Only meaningful after an actual (non-dry-run) write.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

VERIFY_N = 12
MERGED_COLORS = _class_colors(len(MERGED9_NAMES))
MERGED_PERSON_ID = MERGED9_NAMES.index("person")

updated_rows = [r for r in manifest_rows
                if (FULL_OUT_DIR / ORIGINAL_SPLIT_TO_OURS[r["original_split"]] / "labels"
                    / (Path(r["original_filename"]).stem + ".txt")).exists()]
verify_sample = random.sample(updated_rows, min(VERIFY_N, len(updated_rows)))


def draw_merged(image, rows):
    img = image.convert("RGB").copy()
    draw = ImageDraw.Draw(img)
    w_img, h_img = img.size
    for cls_id, cx, cy, w, h in rows:
        x1, y1 = (cx - w / 2) * w_img, (cy - h / 2) * h_img
        x2, y2 = (cx + w / 2) * w_img, (cy + h / 2) * h_img
        color = PERSON_COLOR if cls_id == MERGED_PERSON_ID else MERGED_COLORS[cls_id]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1 + 2, y1 + 2), MERGED9_NAMES[cls_id], fill=color)
    return img


fig, axes = plt.subplots(3, 4, figsize=(20, 14))
for ax, row in zip(axes.flat, verify_sample):
    img_path = merged_split_dir(row["split"], "images") / row["merged_filename"]
    label_path = merged_split_dir(row["split"], "labels") / (Path(row["merged_filename"]).stem + ".txt")
    rows = read_yolo_labels(label_path)
    image = Image.open(img_path)
    ax.imshow(draw_merged(image, rows))
    ax.set_title(f"{row['split']}/{row['merged_filename']}", fontsize=7)
    ax.axis("off")

for ax in axes.flat[len(verify_sample):]:
    ax.axis("off")

plt.tight_layout()
plt.show()